In [1]:
# --- RandomForest training, mirroring train_30feat.py ---
# Same training_data.parquet, same stratified 80/20 split with
# random_state=42, same top-30 features by gain from the 319-feat XGB
# model. Wrapped in a Pipeline because RandomForest can't ingest NaN
# natively (boosting models can).

import json
import shutil
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

CACHE = Path("cache")
TRAIN_PARQUET = CACHE / "training_data.parquet"
MODEL_DIR = CACHE / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

EXISTING_MODEL = MODEL_DIR / "xgb_insider_latest.json"
EXISTING_META  = MODEL_DIR / "xgb_insider_latest.meta.json"

TOP_K = 30
LABEL_COL = "is_insider"
THRESHOLD = 0.5
RANDOM_STATE = 42

MODEL_TAG = "rf_insider_30feat"

In [2]:
print(f"loading {TRAIN_PARQUET}")
df = pd.read_parquet(TRAIN_PARQUET)
print(f"shape={df.shape}")
print(f"positives (insiders) = {int((df[LABEL_COL] == 1).sum())}")
print(f"negatives            = {int((df[LABEL_COL] == 0).sum())}")

loading cache/training_data.parquet
shape=(793, 322)
positives (insiders) = 117
negatives            = 676


In [3]:
with open(EXISTING_META) as f:
    base_meta = json.load(f)
feature_names = base_meta["features"]

xgb_booster = xgb.XGBClassifier()
xgb_booster.load_model(str(EXISTING_MODEL))
importances = xgb_booster.feature_importances_

if len(importances) != len(feature_names):
    raise SystemExit(
        f"importance length {len(importances)} != feature names {len(feature_names)}"
    )

ranked = sorted(zip(feature_names, importances), key=lambda kv: kv[1], reverse=True)
feature_cols = [name for name, _ in ranked[:TOP_K]]

print(f"top {TOP_K} features by gain from 319-feat XGB:")
for i, (name, imp) in enumerate(ranked[:TOP_K], 1):
    print(f"  {i:2d}. {imp:7.4f}  {name}")

missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise SystemExit(f"missing feature columns in training data: {missing}")

top 30 features by gain from 319-feat XGB:
   1.  0.1624  adaptive_markout_raw_grosswt_h0p070
   2.  0.0619  unrealized_edge_per_open_token
   3.  0.0598  adaptive_markout_raw_grosswt_h0p010
   4.  0.0413  adaptive_markout_raw_grosswt_h0p150
   5.  0.0311  unrealized_edge_per_open_token_sqrt
   6.  0.0294  buy_markout_adj_grosswt_h0p070_lb0p150
   7.  0.0175  adaptive_markout_raw_grosswt_h0p200
   8.  0.0153  buy_markout_adj_grosswt_h0p030_lb0p200
   9.  0.0147  high_sell_nonflattening_cash_frac_pctl90
  10.  0.0135  realized_pnl_per_closing_trade
  11.  0.0133  adaptive_plus_realized_alpha0p25_h0p030
  12.  0.0129  buy_markout_adj_grosswt_h0p030_lb0p010
  13.  0.0122  adaptive_plus_realized_alpha0p5_h0p070
  14.  0.0115  buy_markout_adj_mean_h0p200_lb0p030
  15.  0.0106  buy_markout_adj_mean_h0p200_lb0p200
  16.  0.0102  sell_markout_adj_median_h0p150_lb0p010
  17.  0.0098  n_distinct_buy_days
  18.  0.0095  major_move_time_remaining_pct
  19.  0.0094  time_to_major_move_pct_of_market

In [4]:
X = df[feature_cols].astype(float)
y = df[LABEL_COL].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
)

n_pos = int((y_train == 1).sum())
n_neg = int((y_train == 0).sum())
scale_pos_weight = n_neg / max(n_pos, 1)

print(f"train n={len(X_train)} (pos={n_pos}, neg={n_neg})  "
      f"scale_pos_weight={scale_pos_weight:.3f}")
print(f"test  n={len(X_test)} "
      f"(pos={int((y_test == 1).sum())}, neg={int((y_test == 0).sum())})")

train n=634 (pos=94, neg=540)  scale_pos_weight=5.745
test  n=159 (pos=23, neg=136)


In [5]:
# --- RandomForest with hyperparams suited to the model ---
# Boosting uses shallow trees (max_depth=4); RF works best with deep
# trees averaged together, so we leave max_depth=None and use 300 trees
# matching the boosting n_estimators count for a fair compute budget.
# class_weight = {0: 1, 1: scale_pos_weight} replicates XGB's class
# weighting exactly.

rf_params = dict(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight={0: 1.0, 1: scale_pos_weight},
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf",     RandomForestClassifier(**rf_params)),
])

t0 = time.time()
model.fit(X_train, y_train)
print(f"fit in {time.time() - t0:.2f}s")

fit in 0.64s


In [6]:
proba = model.predict_proba(X_test)[:, 1]
pred  = (proba >= THRESHOLD).astype(int)

roc_auc = roc_auc_score(y_test, proba)
pr_auc  = average_precision_score(y_test, proba)

print(f"TEST ROC-AUC = {roc_auc:.4f}")
print(f"TEST PR-AUC  = {pr_auc:.4f}")
print()
print(f"confusion matrix @ thr={THRESHOLD}")
print(confusion_matrix(y_test, pred))
print()
print(classification_report(y_test, pred, digits=4))

print("RandomForest feature importances (mean decrease in impurity):")
rf = model.named_steps["clf"]
for name, imp in sorted(zip(feature_cols, rf.feature_importances_),
                        key=lambda kv: kv[1], reverse=True):
    print(f"  {imp:7.4f}  {name}")

TEST ROC-AUC = 0.9340
TEST PR-AUC  = 0.8348

confusion matrix @ thr=0.5
[[132   4]
 [  6  17]]

              precision    recall  f1-score   support

           0     0.9565    0.9706    0.9635       136
           1     0.8095    0.7391    0.7727        23

    accuracy                         0.9371       159
   macro avg     0.8830    0.8549    0.8681       159
weighted avg     0.9353    0.9371    0.9359       159

RandomForest feature importances (mean decrease in impurity):
   0.0928  unrealized_edge_per_open_token
   0.0764  unrealized_edge_per_open_token_sqrt
   0.0749  time_to_major_move_pct_of_market
   0.0731  adaptive_markout_raw_grosswt_h0p010
   0.0703  adaptive_markout_raw_grosswt_h0p070
   0.0676  adaptive_markout_raw_grosswt_h0p030
   0.0610  adaptive_markout_raw_grosswt_h0p200
   0.0606  adaptive_markout_raw_grosswt_h0p150
   0.0393  buy_markout_adj_grosswt_h0p030_lb0p200
   0.0392  buy_markout_adj_grosswt_h0p030_lb0p010
   0.0375  buy_markout_adj_grosswt_h0p070_lb0p1

In [7]:
ts = time.strftime("%Y%m%d_%H%M%S")
model_path = MODEL_DIR / f"{MODEL_TAG}_{ts}.joblib"
meta_path  = MODEL_DIR / f"{MODEL_TAG}_{ts}.meta.json"

joblib.dump(model, model_path)

meta = {
    "trained_at": ts,
    "model_type": "sklearn.Pipeline[SimpleImputer, RandomForestClassifier]",
    "n_train": len(X_train),
    "n_test": len(X_test),
    "n_features": len(feature_cols),
    "features": feature_cols,
    "feature_selection": (
        f"top {TOP_K} by feature_importances_ (gain) from {EXISTING_MODEL.name}"
    ),
    "scale_pos_weight": scale_pos_weight,
    "threshold": THRESHOLD,
    "model_params": rf_params,
    "metrics": {"roc_auc": float(roc_auc), "pr_auc": float(pr_auc)},
    "source_training_data": str(TRAIN_PARQUET),
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

latest_model = MODEL_DIR / f"{MODEL_TAG}_latest.joblib"
latest_meta  = MODEL_DIR / f"{MODEL_TAG}_latest.meta.json"
shutil.copy(model_path, latest_model)
shutil.copy(meta_path, latest_meta)

print(f"saved {model_path}")
print(f"saved {meta_path}")
print(f"copied -> {latest_model}")
print(f"copied -> {latest_meta}")

saved cache/models/rf_insider_30feat_20260529_140510.joblib
saved cache/models/rf_insider_30feat_20260529_140510.meta.json
copied -> cache/models/rf_insider_30feat_latest.joblib
copied -> cache/models/rf_insider_30feat_latest.meta.json


In [8]:
def _load_metrics(meta_file: Path) -> dict | None:
    try:
        with open(meta_file) as f:
            return json.load(f).get("metrics", {})
    except FileNotFoundError:
        return None

rows = []
for tag, path in [
    ("xgb  319-feat",  MODEL_DIR / "xgb_insider_latest.meta.json"),
    ("xgb   50-feat",  MODEL_DIR / "xgb_insider_50feat_latest.meta.json"),
    ("xgb   30-feat",  MODEL_DIR / "xgb_insider_30feat_latest.meta.json"),
    ("xgb   14-feat",  MODEL_DIR / "xgb_insider_14feat_latest.meta.json"),
    ("lgbm  30-feat",  MODEL_DIR / "lgbm_insider_30feat_latest.meta.json"),
    ("cb    30-feat",  MODEL_DIR / "cb_insider_30feat_latest.meta.json"),
]:
    m = _load_metrics(path)
    if m:
        rows.append((tag, m.get("roc_auc", 0.0), m.get("pr_auc", 0.0)))
rows.append(("rf    30-feat", roc_auc, pr_auc))

print(f"{'model':<16}  {'ROC-AUC':>8}  {'PR-AUC':>8}")
print("-" * 38)
for tag, ra, pa in rows:
    print(f"{tag:<16}  {ra:>8.4f}  {pa:>8.4f}")

model              ROC-AUC    PR-AUC
--------------------------------------
xgb  319-feat       0.9492    0.8399
xgb   50-feat       0.9341    0.8305
xgb   30-feat       0.9469    0.8362
xgb   14-feat       0.8900    0.6744
lgbm  30-feat       0.9511    0.8459
cb    30-feat       0.9578    0.8893
rf    30-feat       0.9340    0.8348
